In [ ]:
import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# 1. Config
# Update this to match the Kaggle dataset path in your notebook
DATA_DIR = "/kaggle/input/the-fake-or-real-dataset/for-norm/for-norm/training"
MAX_DURATION = 3  # seconds
SAMPLE_RATE = 22050
MAX_FILES_PER_CLASS = 500  # limit for free-tier memory safety

# 2. Feature Extraction Function
def extract_spectrogram(file_path):
    try:
        # Load audio, trim to MAX_DURATION
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=MAX_DURATION)
        # Pad if too short
        target_len = MAX_DURATION * SAMPLE_RATE
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        # Generate Mel Spectrogram
        mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_spect = librosa.power_to_db(mel_spect, ref=np.max)
        # Add channel dimension for CNN
        return mel_spect[..., np.newaxis]
    except Exception as e:
        print("Error loading:", file_path, e)
        return None

# 3. Load Data
X = []
y = []

# Assuming folders: 'real' and 'fake' inside DATA_DIR
for label, folder in enumerate(["fake", "real"]):  # 0=Fake, 1=Real
    folder_path = os.path.join(DATA_DIR, folder)
    if not os.path.exists(folder_path):
        print("Missing folder:", folder_path)
        continue

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(".wav")]
    files = files[:MAX_FILES_PER_CLASS]
    print(folder, "files:", len(files))

    for file in files:
        path = os.path.join(folder_path, file)
        features = extract_spectrogram(path)
        if features is not None:
            X.append(features)
            y.append(label)
    print("Total samples loaded so far:", len(X))

X = np.array(X)
y = np.array(y)

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Build CNN Model
model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 6. Train
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

# 7. Save
model.save('deepfake_audio_model.h5')
print('Audio model saved!')

In [ ]:
# Export audio model in multiple formats for serving compatibility
import tensorflow as tf
try:
    import keras as k3
    keras_version = k3.__version__
except Exception:
    k3 = None
    keras_version = 'not-installed'
print('TF version:', tf.__version__)
print('Keras (standalone) version:', keras_version)

# 1) H5 (no optimizer) – good for tf.keras.load_model(..., compile=False)
model.save('deepfake_audio_model.h5', include_optimizer=False)
print('Saved H5 -> deepfake_audio_model.h5')

# 2) Keras v3 native format (.keras) – recommended if using standalone keras>=3
try:
    model.save('deepfake_audio_model.keras', include_optimizer=False)
    print('Saved Keras format -> deepfake_audio_model.keras')
except Exception as e:
    print('Saving .keras format failed:', e)

# 3) TensorFlow SavedModel directory – robust across TF versions
try:
    model.save('deepfake_audio_model_saved', save_format='tf')
    print('Saved SavedModel dir -> deepfake_audio_model_saved/')
except Exception as e:
    print('Saving SavedModel format failed:', e)

# Tip: upload one of these to backend/models and restart the backend.